In [1]:
from cassandra.cluster import Cluster
import mysql.connector
import pandas as pd

In [2]:
cnx = mysql.connector.connect(
  host="localhost",
  user="root",
  password="changeme",
  database="pipe"
)

cursor = cnx.cursor(buffered=True)

In [3]:
query="SELECT * FROM track"
cursor.execute(query)

In [4]:
rows = cursor.fetchall()
len(rows)

44190

In [5]:
df = pd.DataFrame(rows, columns=[x[0] for x in cursor.description])

In [6]:
df

,track_id,track_name,artists,album_name,popularity,duration_ms
0,0000vdREvCVMxbQTkS888c,Lolly,Rill,Lolly,44,160725
1,000CC8EParg64OmTxVnZ0p,It's All Coming Back To Me Now (Glee Cast Vers...,Glee Cast,Glee Love Songs,47,322933
2,0017XiMkqbTfF2AUOzlhj6,Thanksgiving Chicken,Chad Daniels,Busy Being Awesome,24,127040
3,001APMDOl3qtx1526T11n1,Better,Pink Sweat$;Kirby,New RnB,0,176320
4,001pyq8FLNSL1C8orNLI0b,Poor Man,Old Crow Medicine Show,O.C.M.S.,30,214600
...,...,...,...,...,...,...
44185,7zxpdh3EqMq2JCkOI0EqcG,"Two Worlds (From ""Tarzan"")",Piano Genie,Disney Favourites,23,109573
44186,7zXW8qC4ZFmA542eEHppke,Trip To Mars (Astronauts),Sub Zero Project,Renaissance Of Rave,54,182168
44187,7zY46aSULMG2pDG2fMlaI7,Only The Beginning of The Adventure,Harry Gregson-Williams,"The Chronicles of Narnia: The Lion, The Witch ...",54,332373
44188,7zybSU9tFO9HNlwmGF7stc,Sunset Drive,Stereoclip,Echoes,54,234300


In [7]:
session=Cluster(['localhost'], port=9042).connect()

In [8]:
session.execute("""CREATE KEYSPACE IF NOT EXISTS pipe WITH replication = {'class': 'SimpleStrategy', 'replication_factor':1};""")
session.execute('USE pipe')

In [9]:
session.execute("CREATE TABLE IF NOT EXISTS track (track_id VARCHAR PRIMARY KEY, track_name VARCHAR, artists VARCHAR, album_name VARCHAR, popularit INT, duration_ms BIGINT)")

In [10]:
pstatemen = "INSERT INTO track (track_id, track_name, artists, album_name, popularit, duration_ms) VALUES (%s, %s, %s, %s, %s, %s)"

values= []
for i in df.index:
    for c in df.columns:
        values.append(df.loc[i,c])

    session.execute(pstatemen, values)
    values.clear()

In [11]:
result=session.execute("SELECT * FROM track LIMIT 5")
for r in result: print(r)

Row(track_id='6EsYCJD4IF4TsDZcRvesK6', album_name="Reason's Why (The Very Best)", artists='Nickel Creek', duration_ms=231840, popularit=23, track_name='You Don’t Have To Move That Mountain - Live From The Freight And Salvage/2000')
Row(track_id='3niz4765xdQsl0DSMf4DIx', album_name='Rituals', artists='Rotting Christ', duration_ms=283380, popularit=44, track_name='Ze Nigmar')
Row(track_id='3lNJ56GfiNeYRDqkP9c9Ol', album_name='Serenity', artists='Little Symphony', duration_ms=99047, popularit=27, track_name='Boyoma')
Row(track_id='72iWI2iHgKIQ8UqQm38g74', album_name='Ten Days in the Madhouse', artists='Denise Ho', duration_ms=235116, popularit=28, track_name='青山黛瑪')
Row(track_id='31IHYkeWBXpiVe5ihGyzje', album_name='The Hardcore Archive Part 2 (1995 - 1996)', artists='The Prophet', duration_ms=354400, popularit=4, track_name='I Love You (Rave Mix) - Extended Mix')
